# ⚡ Generación de Imágenes con SDXL-Turbo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mattbarreto/ifts24-lab-pdi-2026/blob/master/009%20-%20modelos_difusion/04_Text_to_Image_SDXL_Turbo.ipynb)

**Procesamiento Digital de Imágenes — IFTS24**
Prof. Matias Barreto — 2026

**Unidad 9 · Bloque 4 — 60 minutos**

---

## Al terminar este bloque vas a poder:

1. Cargar y configurar el pipeline de SDXL-Turbo desde Hugging Face en Colab.
2. Generar imágenes de 1024×1024 píxeles en un único paso de inferencia.
3. Controlar la reproducibilidad de los resultados usando seeds fijos.
4. Experimentar con distintos prompts y resoluciones para entender los límites del modelo.

---

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Prompt** | Descripción en texto que guía al modelo sobre qué imagen generar; cuanto más preciso, más predecible el resultado. |
| **guidance_scale** | Parámetro que controla cuánto el modelo sigue el prompt; en SDXL-Turbo siempre debe ser 0.0 por diseño del modelo. |
| **Inference steps** | Cantidad de pasos de denoising; SDXL-Turbo opera con exactamente 1 paso (vs. 20-50 de modelos estándar). |
| **Seed** | Número entero que inicializa la aleatoriedad; el mismo seed con el mismo prompt produce siempre la misma imagen. |
| **VRAM** | Memoria dedicada de la GPU; SDXL-Turbo requiere mínimo 6 GB para generar a 1024×1024 sin optimizaciones. |
| **Destilación adversarial** | Técnica de entrenamiento que comprimió 30 pasos de generación de SDXL en solo 1 sin pérdida visible de calidad. |

---

## 1. Configuración Inicial

In [ ]:
# Instalación de dependencias
# Ejecutá esto solo la primera vez o si tenés errores de importación

!pip install diffusers transformers accelerate torch pillow -q

In [ ]:
# Imports necesarios
import torch
from diffusers import AutoPipelineForText2Image
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display
import time
import warnings
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Detección de hardware disponible

device = "cuda" if torch.cuda.is_available() else "cpu"

print("="*50)
print("INFORMACIÓN DEL HARDWARE")
print("="*50)

if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)

    print(f"✓ GPU detectada: {gpu_name}")
    print(f"✓ VRAM disponible: {gpu_memory:.1f} GB")

    if gpu_memory >= 6:
        print("\n✓ Tu GPU tiene suficiente memoria para SDXL-Turbo")
    else:
        print("\n⚠ Tu GPU tiene poca memoria. Vamos a aplicar optimizaciones.")
else:
    print("⚠ No se detectó GPU")
    print("El modelo va a correr en CPU (muy lento)")
    print("Recomendación: Usar Google Colab con GPU habilitada")

print("="*50)

---

## 2. Cargar el Modelo

Vas a cargar SDXL-Turbo desde Hugging Face. La primera vez va a descargar ~6.9 GB.

In [ ]:
# Cargar el modelo SDXL-Turbo

MODEL_ID = "stabilityai/sdxl-turbo"

print(f"Cargando modelo: {MODEL_ID}")
print("Esto puede tardar 2-5 minutos la primera vez (descarga del modelo)...\n")

pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    variant="fp16" if device == "cuda" else None
)

pipe = pipe.to(device)

print("✓ Modelo cargado exitosamente")

### Optimizaciones de Memoria (Opcional)

Si tenés problemas de memoria (error "CUDA out of memory"), ejecutá la siguiente celda:

In [ ]:
# Optimizaciones para GPUs con poca memoria
# Ejecutá esto solo si tenés errores de memoria

if device == "cuda":
    # Mueve componentes entre GPU y CPU según necesidad
    pipe.enable_model_cpu_offload()

    # Procesa VAE en tiles para imágenes grandes
    pipe.enable_vae_tiling()

    print("✓ Optimizaciones de memoria habilitadas")
    print("Esto hace el proceso un poco más lento pero usa menos VRAM")

---

## 3. Primera Generación

Vas a generar tu primera imagen con un prompt simple.

In [ ]:
# Primera generación con prompt simple

prompt = "a beautiful sunset over mountains, vibrant colors, highly detailed"

print(f"Generando: '{prompt}'\n")

# Medir tiempo
start_time = time.time()

# Generar imagen
image = pipe(
    prompt=prompt,
    num_inference_steps=1,     # Turbo usa 1 paso
    guidance_scale=0.0,         # Turbo NO usa guidance
).images[0]

end_time = time.time()
tiempo_generacion = end_time - start_time

# Mostrar resultado
display(image)
print(f"\n✓ Imagen generada en {tiempo_generacion:.2f} segundos")
print(f"Tamaño: {image.size[0]} x {image.size[1]} píxeles")

### ¿Qué Acabó de Pasar?

1. El modelo tomó tu **prompt** (descripción en texto)
2. Generó una imagen de **1024x1024 píxeles**
3. Lo hizo en **~1 segundo** (en GPU)
4. Usó **un solo paso** de inferencia (vs 20-50 de modelos normales)

Esto es posible gracias a **Adversarial Diffusion Distillation**, una técnica que comprime el conocimiento de modelos grandes en modelos que generan en 1 paso.

---

## 4. Experimentar con Prompts

La calidad de la imagen depende mucho de cómo escribas el prompt.

In [ ]:
# Función para generar y mostrar múltiples imágenes

def generar_y_mostrar(prompt, mostrar_tiempo=True):
    """
    Genera una imagen a partir de un prompt y la muestra.

    Args:
        prompt (str): Descripción de la imagen a generar
        mostrar_tiempo (bool): Si mostrar el tiempo de generación

    Returns:
        PIL.Image: Imagen generada
    """
    start = time.time()

    imagen = pipe(
        prompt=prompt,
        num_inference_steps=1,
        guidance_scale=0.0
    ).images[0]

    tiempo = time.time() - start

    print(f"Prompt: '{prompt}'")
    if mostrar_tiempo:
        print(f"Tiempo: {tiempo:.2f}s")
    print()

    display(imagen)
    return imagen

print("Función lista para usar")

In [ ]:
# Ejemplo 1: Fotografía realista

prompt_foto = "a professional portrait photo of a young woman, natural lighting, sharp focus, detailed, high quality"

imagen1 = generar_y_mostrar(prompt_foto)

In [ ]:
# Ejemplo 2: Paisaje natural

prompt_paisaje = "a serene lake surrounded by pine trees, mountains in background, golden hour lighting, photographic style"

imagen2 = generar_y_mostrar(prompt_paisaje)

In [ ]:
# Ejemplo 3: Arte digital

prompt_arte = "a futuristic cyberpunk city at night, neon lights, digital art, highly detailed, vibrant colors"

imagen3 = generar_y_mostrar(prompt_arte)

In [ ]:
# Ejemplo 4: Objeto/Producto

prompt_objeto = "a modern smartphone on a marble table, product photography, studio lighting, sharp focus"

imagen4 = generar_y_mostrar(prompt_objeto)

### Ejercicio: Tu Turno

Probá escribir tus propios prompts. Algunos tips:

**Estructura básica:**
```
[sujeto], [detalles], [estilo], [iluminación], [calidad]
```

**Ejemplos de términos útiles:**
- Estilos: "photographic", "digital art", "oil painting", "watercolor"
- Calidad: "highly detailed", "sharp focus", "high quality", "8k"
- Iluminación: "natural lighting", "golden hour", "studio lighting", "dramatic lighting"

Ver más tips en: `06_Material_Apoyo/Guia_Prompts_Efectivos.md`

In [ ]:
# TU PROMPT AQUÍ
# Modificá esta línea con tu propia descripción

mi_prompt = ""  # 👈 Escribí tu prompt entre las comillas

if mi_prompt:
    mi_imagen = generar_y_mostrar(mi_prompt)
else:
    print("Escribí un prompt en la variable 'mi_prompt' y ejecutá de nuevo")

---

## 5. Generar Múltiples Variaciones

Podés generar varias imágenes del mismo prompt para ver diferentes interpretaciones.

In [ ]:
# Generar 4 variaciones del mismo prompt

prompt_variaciones = "a cute orange cat sleeping on a couch"

print(f"Generando 4 variaciones de: '{prompt_variaciones}'\n")

imagenes = []
for i in range(4):
    img = pipe(
        prompt=prompt_variaciones,
        num_inference_steps=1,
        guidance_scale=0.0
    ).images[0]
    imagenes.append(img)
    print(f"✓ Variación {i+1}/4 generada")

# Mostrar en grid
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for idx, (img, ax) in enumerate(zip(imagenes, axes.flat)):
    ax.imshow(img)
    ax.set_title(f"Variación {idx+1}")
    ax.axis('off')
plt.tight_layout()
plt.show()

### Observación

Notá que aunque el prompt es el mismo, cada generación es única. Esto es porque:
1. El modelo parte de **ruido aleatorio** diferente cada vez
2. Hay aleatoriedad inherente en el proceso de difusión

Para obtener resultados reproducibles, podés usar un **seed fijo** (lo vemos más adelante).

---

## 6. Control de Tamaño de Imagen

Por defecto, SDXL-Turbo genera imágenes de 1024x1024. Podés cambiar esto.

In [ ]:
# Generar en diferentes resoluciones

prompt_res = "a modern house with large windows, architectural photography"

resoluciones = [
    (512, 512, "Baja (512x512)"),
    (768, 768, "Media (768x768)"),
    (1024, 1024, "Alta (1024x1024)")
]

print(f"Generando en 3 resoluciones diferentes...\n")

imgs_res = []
for width, height, label in resoluciones:
    img = pipe(
        prompt=prompt_res,
        num_inference_steps=1,
        guidance_scale=0.0,
        height=height,
        width=width
    ).images[0]
    imgs_res.append((img, label))
    print(f"✓ {label} generada")

# Mostrar comparación
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for (img, label), ax in zip(imgs_res, axes):
    ax.imshow(img)
    ax.set_title(label)
    ax.axis('off')
plt.tight_layout()
plt.show()

print("\nNota: Resoluciones más altas usan más VRAM pero tienen más detalle")

**Recomendaciones de resolución:**
- **512x512**: Para testear rápido o GPUs con <4GB VRAM
- **768x768**: Balance entre calidad y memoria
- **1024x1024**: Máxima calidad (requiere ~6GB VRAM)
- **Formatos no cuadrados**: También funcionan (ej: 1024x768)

---

## 7. Reproducibilidad con Seeds

Para obtener la misma imagen cada vez, usá un seed fijo.

In [ ]:
# Función para generar con seed específico

def generar_con_seed(prompt, seed):
    """
    Genera imagen con seed fijo para reproducibilidad.

    Args:
        prompt (str): Descripción de la imagen
        seed (int): Semilla aleatoria (mismo seed = misma imagen)

    Returns:
        PIL.Image: Imagen generada
    """
    # Configurar generator con seed
    generator = torch.Generator(device=device).manual_seed(seed)

    imagen = pipe(
        prompt=prompt,
        num_inference_steps=1,
        guidance_scale=0.0,
        generator=generator
    ).images[0]

    return imagen

print("Función lista para usar")

In [ ]:
# Demostración: mismo seed = misma imagen

prompt_seed = "a red sports car in a garage"
seed_fijo = 42

print(f"Generando 3 veces con seed={seed_fijo}\n")

imagenes_seed = []
for i in range(3):
    img = generar_con_seed(prompt_seed, seed_fijo)
    imagenes_seed.append(img)
    print(f"✓ Generación {i+1}/3")

# Mostrar (deberían ser idénticas)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx, (img, ax) in enumerate(zip(imagenes_seed, axes)):
    ax.imshow(img)
    ax.set_title(f"Iteración {idx+1} (seed={seed_fijo})")
    ax.axis('off')
plt.tight_layout()
plt.show()

print("\nObservá que las 3 imágenes son idénticas (mismo seed)")

In [ ]:
# Comparación: diferentes seeds

print("Ahora con seeds diferentes:\n")

seeds = [42, 123, 999]
imagenes_diferentes = []

for seed in seeds:
    img = generar_con_seed(prompt_seed, seed)
    imagenes_diferentes.append(img)
    print(f"✓ Generada con seed={seed}")

# Mostrar
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for seed, img, ax in zip(seeds, imagenes_diferentes, axes):
    ax.imshow(img)
    ax.set_title(f"seed={seed}")
    ax.axis('off')
plt.tight_layout()
plt.show()

print("\nCon seeds diferentes, obtenés variaciones del mismo prompt")

**Cuándo usar seeds:**
- Para **reproducir** exactamente una imagen que te gustó
- Para **comparar** el efecto de cambiar solo el prompt
- Para **debugging** (aislar si el problema es el modelo o el prompt)
- En **producción** si necesitás consistencia

---

## 8. Guardar Imágenes Generadas

In [ ]:
# Guardar imagen en disco

# Generar imagen
prompt_guardar = "a beautiful flower in a garden, macro photography"
imagen_final = pipe(
    prompt=prompt_guardar,
    num_inference_steps=1,
    guidance_scale=0.0
).images[0]

# Guardar
nombre_archivo = "mi_imagen_generada.png"
imagen_final.save(nombre_archivo)

print(f"✓ Imagen guardada como: {nombre_archivo}")
display(imagen_final)

In [ ]:
# Guardar con metadata (incluir el prompt en la imagen)

from PIL.PngImagePlugin import PngInfo

prompt_metadata = "a sunset landscape"
imagen_meta = pipe(prompt=prompt_metadata, num_inference_steps=1, guidance_scale=0.0).images[0]

# Crear metadata
metadata = PngInfo()
metadata.add_text("prompt", prompt_metadata)
metadata.add_text("model", "SDXL-Turbo")
metadata.add_text("steps", "1")

# Guardar con metadata
imagen_meta.save("imagen_con_metadata.png", pnginfo=metadata)

print("✓ Imagen guardada con metadata")
print("La metadata incluye el prompt usado")

---

## 9. Limitaciones y Consideraciones

### Qué Funciona Bien

- Fotografías realistas
- Paisajes naturales
- Objetos y productos
- Arte digital
- Escenas generales

### Limitaciones Conocidas

- **Texto en imágenes:** No genera texto legible correctamente
- **Anatomía compleja:** Manos, pies pueden tener errores
- **Conceptos muy específicos:** Puede no entender prompts muy técnicos
- **Consistencia:** Difícil mantener el mismo personaje en múltiples generaciones

### Comparado con Modelos Base

**SDXL-Turbo vs SDXL base:**
- ✓ Mucho más rápido (1 paso vs 30-50)
- ✓ Misma calidad general
- ✗ Menos control fino (no podés ajustar steps o guidance)
- ✗ No funciona con LoRAs custom

### Uso Ético y Legal

- Revisar licencia para uso comercial
- No generar contenido dañino o ilegal
- Ser transparente sobre uso de IA generativa
- Respetar derechos de autor y marcas

---

## 10. Ejercicios Prácticos

### Ejercicio 1: Categorías

Generá una imagen para cada categoría:
1. Retrato fotográfico
2. Paisaje natural
3. Objeto/producto
4. Arte digital/ilustración
5. Arquitectura

Guardá tus mejores resultados.

In [ ]:
# Tu código para Ejercicio 1

### Ejercicio 2: Exploración de Seeds

Elegí un prompt y probá con 5 seeds diferentes (ej: 1, 100, 500, 1000, 5000).
¿Qué variaciones observás?

In [ ]:
# Tu código para Ejercicio 2

### Ejercicio 3: Optimización del Prompt

Tomá un concepto simple (ej: "perro") y mejoralo iterativamente:
1. "a dog"
2. "a golden retriever dog"
3. "a golden retriever dog sitting in a park"
4. "a golden retriever dog sitting in a park, sunny day, professional photography"

Compará los resultados.

In [ ]:
# Tu código para Ejercicio 3

### ✎ Para pensar

1. Si usás el mismo seed en dos sesiones distintas de Colab, ¿obtenés la misma imagen? ¿De qué factores depende?
2. ¿Para qué caso de uso profesional es crítico usar seeds fijos? Pensá en al menos dos ejemplos concretos.

### ✎ Para pensar

1. SDXL-Turbo genera en 1 solo paso lo que otros modelos hacen en 30-50. ¿Qué información tuvo que «preaprender» para poder hacer eso?
2. `guidance_scale=0.0` significa que el modelo no usa la señal del texto durante la generación. ¿Cómo puede entonces seguir el prompt?

---

## Próximos Pasos

Ahora que dominás SDXL-Turbo, podés:

1. **Explorar otros modelos:**
   - `05_Text_to_Image_SDXS_CPU.ipynb` - Máxima calidad
   - `05_Text_to_Image_SDXS_CPU.ipynb` - Optimizado para CPU
   - `06_Aceleracion_LCM_LoRA.ipynb` - Acelerar modelos existentes

2. **Aplicaciones prácticas:**
   - `03_Aplicaciones_Practicas_Difusion.ipynb`
   - `03_Aplicaciones_Practicas_Difusion.ipynb`

3. **Optimizaciones avanzadas:**
   - `los materiales de la carpeta `archivo/``

---


## ⛰️ Lo que construimos

| Concepto | Lo que aprendimos |
|---|---|
| SDXL-Turbo | Modelo destilado que genera en 1 paso con calidad comparable a 30 |
| guidance_scale=0.0 | Configuración obligatoria; no es un error, es el diseño del modelo |
| Seeds | Herramienta de reproducibilidad para controlar la aleatoriedad del proceso |
| Resoluciones | 512×512 (rápido) → 768×768 (balance) → 1024×1024 (máxima calidad) |

**Próximo cuaderno:** `05_Text_to_Image_SDXS_CPU` — generamos imágenes sin GPU usando SDXS-512.